In [1]:
import duckdb
import pandas as pd
import numpy as np
from datetime import datetime

# today
today = datetime.today().strftime("%y%m%d")

## Pull publications for manual review

In [2]:
# Connect to database
db = duckdb.connect("publications.db")
db.execute("PRAGMA memory_limit='3GB'")
db.execute("PRAGMA temp_directory='C:/temp/duckdb_spill'")
db.execute("PRAGMA threads=2")

In [3]:
# Show tables in database
db.sql("SHOW TABLES")

┌─────────────────────────┐
│          name           │
│         varchar         │
├─────────────────────────┤
│ publications_classified │
│ publications_embedding  │
│ run_260710_1615         │
│ run_260710_1615_reverse │
│ run_260713_0836         │
│ run_260713_0836_reverse │
└─────────────────────────┘

#### Get data from publications_classified

In [4]:
# Create dataframe from classified publications
data = db.sql("SELECT * FROM publications_classified").df()

In [ ]:
data

scope_LLM
out    5299
in     3098
Name: count, dtype: int64

In [ ]:
# Filter data for publications that went through LLM scoping
# data = data[data['pred_combined'] == 'in']

In [ ]:
# If this this is a review on an updated dataset, meaning hte columns scope_curated and pillar_curated already exist, perform this filtering step
# data = data[data['scope_curated'].isna()]

#### Create filter mask for curated scope and pillar 

In [10]:
# decision mask for manual review
inherit_mask = (
    # scope_LLM=in, confident, and a pillar was assigned
    ((data['scope_LLM'] == 'in') & (data['confidence_LLM'] > 4) & (data['pillar_LLM'] != 'NA')) | 

    # scope_LLM=in, borderline confidence, but ML agrees on the pillar
    ((data['scope_LLM'] == 'in') & (data['confidence_LLM'] == 4) & (data['pillar_LLM'] != 'NA') & (data['pillar_LLM'] == data['pred_pillar'])) | 

    # scope_LLM=out, confident, and no pillar was assigned
    ((data['scope_LLM'] == 'out') & (data['confidence_LLM'] < 2) & (data['pillar_LLM'] == 'NA'))
)

#### Assign curated scope/pillar or manual review

In [ ]:
# Create curated scope and pillar columns with decision criteria
data['scope_curated']  = np.where(inherit_mask, data['scope_LLM'],  'manual_review')
data['scope_curated']  = np.where(data['pred_combined'] == 'out', 'out',  data['scope_curated'])
data['pillar_curated'] = np.where(inherit_mask, data['pillar_LLM'], 'manual_review')
data['pillar_curated'] = np.where(data['pred_combined'] == 'out', np.nan,  data['pillar_curated'])
data['date_review'] = today

In [13]:
# How many publications have to go through manual review?
data['scope_curated'].value_counts()

scope_curated
out              13579
in                3082
manual_review     2777
Name: count, dtype: int64

In [20]:
data[data['scope_curated'] == 'manual_review']['confidence_LLM'].value_counts()

confidence_LLM
2.0    2575
3.0     177
4.0      15
1.0       2
6.0       1
5.0       1
Name: count, dtype: int64

#### Save borderline publications for manual review

In [21]:
# Export publications for review
review = data[data['scope_curated'] == 'manual_review']
review.to_csv(f"data/publications_for_review_{today}.csv")

#### Save (automatically) curated scope and pillar back to publications_classified

In [22]:
# filter for the publications with clear scope and pillar assigment
data = data[(data['scope_curated'] != 'manual_review') & (~data['scope_curated'].isna())]

In [23]:
# create columns in publications_classified if they don't already exist
db.sql(f"ALTER TABLE publications_classified ADD COLUMN IF NOT EXISTS scope_curated VARCHAR")
db.sql(f"ALTER TABLE publications_classified ADD COLUMN IF NOT EXISTS pillar_curated VARCHAR")

In [24]:
# add to database
db.register('data', data[['id', 'scope_curated', 'pillar_curated']])
db.sql(f"""
    UPDATE publications_classified
    SET scope_curated     = data.scope_curated,
        pillar_curated    = data.pillar_curated
    FROM data
    WHERE publications_classified.id = data.id
""")

In [25]:
db.close()

## Assign manually curated scope and pillar to respective publications

In [49]:
# Connect to database
db = duckdb.connect("publications.db")

In [45]:
# load manually reviewed data
reviewed_data = pd.read_csv("data/publications_reviewed_202324_260715.csv")

#### Add scope and pillar back to original dataset, using publication ID

In [42]:
db.sql(f"ALTER TABLE publications_classified ADD COLUMN IF NOT EXISTS date_review VARCHAR")
db.sql(f"ALTER TABLE publications_classified ADD COLUMN IF NOT EXISTS reviewer VARCHAR")

In [46]:
# add to database
db.register('reviewed_data', reviewed_data[['id', 'scope_curated', 'pillar_curated', 'date_review', 'reviewer']])
db.sql(f"""
    UPDATE publications_classified
    SET scope_curated     = reviewed_data.scope_curated,
        pillar_curated    = reviewed_data.pillar_curated,
        date_review       = reviewed_data.date_review,
        reviewer          = reviewed_data.reviewer
    FROM reviewed_data
    WHERE publications_classified.id = reviewed_data.id
""")

In [33]:
data = db.sql("SELECT * FROM publications_classified").df()

In [37]:
data[(data['scope_curated'].isna())]

,id,title,abstract,authors,concepts_relevant,date,open_access,research_org_cities,research_org_countries,research_org_names,...,status_LLM,stop_reason_LLM,date_LLM,scope_curated,pillar_curated,primary_category_LLM,secondary_category_LLM,category_status_LLM,category_stop_reason_LLM,date_labelling


In [51]:
db.close()